# LeetCode Medium-to-Hard Workbook
## Advanced Python Patterns, Algorithms, and Interview Reasoning

**Goal:** Learn to recognize and implement advanced patterns, explain correctness, and analyze complexity.

### How to work through this workbook
1. Read the problem and write down the input constraints.
2. Try a brute-force solution before optimizing.
3. Write a short explanation of the key observation.
4. Implement the algorithm and test edge cases.
5. Explain why it is correct.
6. State time and auxiliary-space complexity.
7. Revisit problems you could not solve independently.

This workbook uses original practice prompts and examples. The “Suggested LeetCode practice” titles are search references; look up the official problem on LeetCode for the exact statement and constraints.

## Roadmap

| Module | Pattern | Typical complexity |
|---|---|---|
| 1 | Monotonic stacks | O(n) |
| 2 | Binary search on the answer | O(n log R) or O(n log n) |
| 3 | Advanced dynamic programming | O(n²), O(nK), or optimized |
| 4 | Graph algorithms | O(V + E), O((V + E) log V) |
| 5 | Union-find and minimum spanning trees | Near-linear / O(E log E) |
| 6 | Shortest paths and state-space search | Depends on graph and weights |
| 7 | Tries, bitmask DP, and interval DP | Often exponential-state or O(n²) |
| 8 | Hard interview patterns and mixed practice | Pattern-dependent |

`n` = number of items, `V` = vertices, `E` = edges, and `R` = numeric search range.

# Module 1 — Monotonic Stacks

A monotonic stack stores items in increasing or decreasing order. When a new value arrives, remove stack items that can no longer be the answer for any future position.

**Use it when the prompt asks for:**
- next greater/smaller value;
- a span or range bounded by larger/smaller values;
- maximum area formed by consecutive bars;
- contributions of each value to subarray minimums/maximums.

## Problem 1 — Largest Rectangle in a Histogram (Hard)

Given nonnegative bar heights, find the largest rectangle area.

**Suggested LeetCode practice:** Largest Rectangle in Histogram.

### Key idea
For each bar, determine how far it can extend left and right before hitting a shorter bar. A monotonic increasing stack can identify these boundaries in one pass. Adding a sentinel zero flushes remaining bars.

In [ ]:
def largest_rectangle_area(heights):
    stack = []  # Indices of bars in nondecreasing height order.
    best = 0
    values = list(heights) + [0]  # Sentinel forces remaining bars to be processed.

    for i, height in enumerate(values):
        while stack and values[stack[-1]] > height:
            bar_height = values[stack.pop()]
            left_smaller_index = stack[-1] if stack else -1
            width = i - left_smaller_index - 1
            best = max(best, bar_height * width)
        stack.append(i)

    return best


assert largest_rectangle_area([2, 1, 5, 6, 2, 3]) == 10
assert largest_rectangle_area([2, 4]) == 4
assert largest_rectangle_area([]) == 0

**Complexity:** O(n) time because every index is pushed and popped at most once; O(n) extra space.

## Problem 2 — Sum of Subarray Minimums (Medium/Hard)

For every nonempty contiguous subarray, find its minimum and return the sum of those minima modulo `1_000_000_007`.

**Suggested LeetCode practice:** Sum of Subarray Minimums.

### Key idea
Instead of enumerating all subarrays, count how many subarrays assign each index as their minimum. Use a consistent tie rule: previous strictly smaller and next smaller-or-equal, or the reverse.

In [ ]:
def sum_subarray_minimums(nums):
    mod = 1_000_000_007
    n = len(nums)
    stack = []
    left_count = [0] * n
    right_count = [0] * n

    # Number of choices for the left boundary; pop greater values.
    for i, value in enumerate(nums):
        while stack and nums[stack[-1]] > value:
            stack.pop()
        left_count[i] = i - stack[-1] if stack else i + 1
        stack.append(i)

    stack.clear()

    # Number of choices for the right boundary; pop greater-or-equal values.
    for i in range(n - 1, -1, -1):
        while stack and nums[stack[-1]] >= nums[i]:
            stack.pop()
        right_count[i] = stack[-1] - i if stack else n - i
        stack.append(i)

    return sum(nums[i] * left_count[i] * right_count[i] for i in range(n)) % mod


assert sum_subarray_minimums([3, 1, 2, 4]) == 17
assert sum_subarray_minimums([11, 81, 94, 43, 3]) == 444

# Module 2 — Binary Search on the Answer

Classic binary search looks for a value in sorted data. **Binary search on the answer** searches a range of candidate answers when feasibility is monotonic.

For example, if a capacity of 20 works, every larger capacity may also work. We can find the smallest feasible capacity.

Checklist:
1. Define the candidate answer range.
2. Write a feasibility function `can(candidate)`.
3. Confirm feasibility is monotonic.
4. Find the first or last feasible candidate.

## Problem 3 — Minimum Capacity to Ship Packages (Medium)

Packages must be shipped in order within `days`. Find the minimum ship capacity that makes this possible.

**Suggested LeetCode practice:** Capacity To Ship Packages Within D Days.

In [ ]:
def ship_within_days(weights, days):
    if not weights or days < 1:
        raise ValueError("weights must be nonempty and days must be positive")

    def can_ship(capacity):
        used_days = 1
        current_load = 0

        for weight in weights:
            if current_load + weight > capacity:
                used_days += 1
                current_load = 0
            current_load += weight

        return used_days <= days

    left, right = max(weights), sum(weights)

    while left < right:
        mid = (left + right) // 2
        if can_ship(mid):
            right = mid
        else:
            left = mid + 1

    return left


assert ship_within_days([1, 2, 3, 1, 1], 4) == 3
assert ship_within_days([3, 2, 2, 4, 1, 4], 3) == 6

**Complexity:** O(n log(sum(weights))) time, O(1) extra space.

## Problem 4 — Split Array to Minimize the Largest Sum (Hard)

Split a list of nonnegative integers into exactly `k` nonempty contiguous groups. Minimize the largest group sum.

**Suggested LeetCode practice:** Split Array Largest Sum.

### Hint
The same greedy feasibility check used for shipping applies: for a candidate maximum sum, count how many groups are needed.

In [ ]:
def split_array_min_largest_sum(nums, k):
    if not nums or not 1 <= k <= len(nums):
        raise ValueError("k must be between 1 and len(nums)")

    def groups_needed(limit):
        groups = 1
        current = 0

        for value in nums:
            if current + value > limit:
                groups += 1
                current = 0
            current += value

        return groups

    left, right = max(nums), sum(nums)

    while left < right:
        mid = (left + right) // 2
        if groups_needed(mid) <= k:
            right = mid
        else:
            left = mid + 1

    return left


assert split_array_min_largest_sum([7, 2, 5, 10, 8], 2) == 18
assert split_array_min_largest_sum([1, 2, 3, 4, 5], 2) == 9

## Problem 5 — Kth Smallest Pair Distance (Hard)

Given integers and `k`, find the `k`th smallest absolute difference among all pairs.

**Suggested LeetCode practice:** Find K-th Smallest Pair Distance.

### Key idea
Sort the numbers. Binary-search a distance `d`; count pairs whose distance is at most `d` using two pointers. This count is monotonic in `d`.

In [ ]:
def kth_smallest_pair_distance(nums, k):
    nums = sorted(nums)
    n = len(nums)
    total_pairs = n * (n - 1) // 2
    if not 1 <= k <= total_pairs:
        raise ValueError("k is outside the valid pair range")

    def count_at_most(distance):
        count = 0
        left = 0
        for right in range(n):
            while nums[right] - nums[left] > distance:
                left += 1
            count += right - left
        return count

    left, right = 0, nums[-1] - nums[0]
    while left < right:
        mid = (left + right) // 2
        if count_at_most(mid) >= k:
            right = mid
        else:
            left = mid + 1
    return left


assert kth_smallest_pair_distance([1, 3, 1], 1) == 0
assert kth_smallest_pair_distance([1, 6, 1], 3) == 5

# Module 3 — Advanced Dynamic Programming

## DP design template

For every DP problem, explicitly write:
- **State:** what does `dp[i]` or `dp[i][j]` mean?
- **Transition:** how is the state built from smaller states?
- **Base cases:** what is already known?
- **Order:** which states must be computed first?
- **Answer:** which state or aggregate is returned?
- **Optimization:** can dimensions be compressed?

## Problem 6 — Coin Change (Medium)

Given coin denominations and a target amount, return the minimum number of coins needed. Return `-1` if impossible.

**Suggested LeetCode practice:** Coin Change.

In [ ]:
def coin_change(coins, amount):
    if amount < 0 or any(c <= 0 for c in coins):
        raise ValueError("amount must be nonnegative and coins positive")

    unreachable = amount + 1
    dp = [unreachable] * (amount + 1)
    dp[0] = 0

    for value in range(1, amount + 1):
        for coin in coins:
            if coin <= value:
                dp[value] = min(dp[value], dp[value - coin] + 1)

    return dp[amount] if dp[amount] != unreachable else -1


assert coin_change([1, 2, 5], 11) == 3
assert coin_change([2], 3) == -1
assert coin_change([1], 0) == 0

**Complexity:** O(amount × number of coin types) time and O(amount) space.

## Problem 7 — Longest Common Subsequence (Medium)

Given two strings, return the length of their longest common subsequence. A subsequence may skip characters but must preserve order.

**Suggested LeetCode practice:** Longest Common Subsequence.

In [ ]:
def longest_common_subsequence(a, b):
    # dp[j] is the LCS length for the processed prefix of a and b[:j].
    dp = [0] * (len(b) + 1)

    for char_a in a:
        previous_diagonal = 0
        for j, char_b in enumerate(b, start=1):
            old_value = dp[j]
            if char_a == char_b:
                dp[j] = previous_diagonal + 1
            else:
                dp[j] = max(dp[j], dp[j - 1])
            previous_diagonal = old_value

    return dp[-1]


assert longest_common_subsequence("abcde", "ace") == 3
assert longest_common_subsequence("abc", "def") == 0

**Complexity:** O(mn) time and O(n) space, where `m` and `n` are string lengths.

## Problem 8 — Edit Distance (Hard)

Return the minimum number of insertions, deletions, and replacements needed to turn one string into another.

**Suggested LeetCode practice:** Edit Distance.

### State
`dp[i][j]` is the minimum edit count to transform the first `i` characters of `word1` into the first `j` characters of `word2`.

In [ ]:
def edit_distance(word1, word2):
    m, n = len(word1), len(word2)
    dp = [[0] * (n + 1) for _ in range(m + 1)]

    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j

    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if word1[i - 1] == word2[j - 1]:
                dp[i][j] = dp[i - 1][j - 1]
            else:
                dp[i][j] = 1 + min(
                    dp[i - 1][j],      # delete
                    dp[i][j - 1],      # insert
                    dp[i - 1][j - 1]   # replace
                )

    return dp[m][n]


assert edit_distance("horse", "ros") == 3
assert edit_distance("intention", "execution") == 5
assert edit_distance("", "abc") == 3

## Problem 9 — Partition Equal Subset Sum (Medium)

Determine whether a list of positive integers can be split into two groups with equal sums.

**Suggested LeetCode practice:** Partition Equal Subset Sum.

### Key idea
If the total is odd, the answer is false. Otherwise, determine whether any subset sums to `total // 2`. Update possible sums in descending order so each number is used at most once.

In [ ]:
def can_partition_equal(nums):
    total = sum(nums)
    if total % 2:
        return False

    target = total // 2
    possible = {0}

    for value in nums:
        # Descending updates prevent reusing the same item in this iteration.
        for subtotal in sorted(possible, reverse=True):
            candidate = subtotal + value
            if candidate == target:
                return True
            if candidate < target:
                possible.add(candidate)

    return target in possible


assert can_partition_equal([1, 5, 11, 5]) is True
assert can_partition_equal([1, 2, 3, 5]) is False
assert can_partition_equal([2, 2]) is True

# Module 4 — Advanced Graph Algorithms

Represent a graph using an adjacency list when the graph is sparse. For a graph with `V` vertices and `E` edges, standard BFS/DFS is O(V + E).

## Problem 10 — Topological Ordering (Medium)

Given a directed acyclic graph, return a valid ordering of vertices such that every prerequisite appears before the dependent vertex. Return an empty list if a cycle exists.

**Suggested LeetCode practice:** Course Schedule II.

In [ ]:
from collections import deque


def topological_order(n, edges):
    graph = [[] for _ in range(n)]
    indegree = [0] * n

    for source, destination in edges:
        graph[source].append(destination)
        indegree[destination] += 1

    queue = deque(i for i in range(n) if indegree[i] == 0)
    order = []

    while queue:
        node = queue.popleft()
        order.append(node)

        for neighbor in graph[node]:
            indegree[neighbor] -= 1
            if indegree[neighbor] == 0:
                queue.append(neighbor)

    return order if len(order) == n else []


order = topological_order(4, [(0, 2), (1, 2), (2, 3)])
assert len(order) == 4
position = {node: i for i, node in enumerate(order)}
assert position[0] < position[2] and position[1] < position[2]
assert topological_order(2, [(0, 1), (1, 0)]) == []

**Complexity:** O(V + E) time and O(V + E) space.

## Problem 11 — Dijkstra's Shortest Paths (Medium/Hard)

Given a directed graph with nonnegative edge weights, compute the shortest distance from a source to every vertex. Use `float("inf")` for unreachable vertices.

**Important:** Dijkstra's algorithm is not correct for graphs with negative edge weights.

In [ ]:
import heapq


def dijkstra(n, edges, source):
    graph = [[] for _ in range(n)]
    for u, v, weight in edges:
        if weight < 0:
            raise ValueError("Dijkstra requires nonnegative edge weights")
        graph[u].append((v, weight))

    distances = [float("inf")] * n
    distances[source] = 0
    heap = [(0, source)]

    while heap:
        distance, node = heapq.heappop(heap)

        # Ignore stale heap entries.
        if distance != distances[node]:
            continue

        for neighbor, weight in graph[node]:
            candidate = distance + weight
            if candidate < distances[neighbor]:
                distances[neighbor] = candidate
                heapq.heappush(heap, (candidate, neighbor))

    return distances


dist = dijkstra(4, [(0, 1, 4), (0, 2, 1), (2, 1, 2), (1, 3, 1)], 0)
assert dist == [0, 3, 1, 4]

**Complexity:** O((V + E) log V) with an adjacency list and binary heap.

## Problem 12 — Bellman-Ford (Hard)

Compute shortest paths when negative edge weights may exist. Detect a reachable negative-weight cycle.

Return `None` if a reachable negative cycle exists; otherwise return the distance list.

**Suggested LeetCode practice:** Cheapest Flights Within K Stops is related but adds a stop constraint; study Bellman-Ford as a general graph algorithm.

In [ ]:
def bellman_ford(n, edges, source):
    dist = [float("inf")] * n
    dist[source] = 0

    # A shortest simple path has at most n - 1 edges.
    for _ in range(n - 1):
        changed = False
        for u, v, weight in edges:
            if dist[u] != float("inf") and dist[u] + weight < dist[v]:
                dist[v] = dist[u] + weight
                changed = True
        if not changed:
            break

    # A further relaxation means a reachable negative cycle exists.
    for u, v, weight in edges:
        if dist[u] != float("inf") and dist[u] + weight < dist[v]:
            return None

    return dist


assert bellman_ford(3, [(0, 1, 4), (1, 2, -2)], 0) == [0, 4, 2]
assert bellman_ford(3, [(0, 1, 1), (1, 2, -2), (2, 1, -2)], 0) is None

## Problem 13 — Multi-Source BFS (Medium)

Given an unweighted graph and several source vertices, find the minimum distance from any source to every vertex.

This pattern appears in grid problems such as distance-to-nearest-zero or spreading processes.

In [ ]:
def multi_source_bfs(n, edges, sources):
    graph = [[] for _ in range(n)]
    for u, v in edges:
        graph[u].append(v)
        graph[v].append(u)

    distance = [-1] * n
    queue = deque()

    for source in sources:
        if distance[source] == -1:
            distance[source] = 0
            queue.append(source)

    while queue:
        node = queue.popleft()
        for neighbor in graph[node]:
            if distance[neighbor] == -1:
                distance[neighbor] = distance[node] + 1
                queue.append(neighbor)

    return distance


assert multi_source_bfs(5, [(0, 1), (1, 2), (2, 3)], [0, 3]) == [0, 1, 1, 0, -1]

# Module 5 — Union-Find and Minimum Spanning Trees

Union-Find (Disjoint Set Union, DSU) maintains connected groups. Path compression and union by size make operations extremely fast in practice.

## Problem 14 — Count Connected Components Using DSU (Medium)

In [ ]:
class DSU:
    def __init__(self, n):
        self.parent = list(range(n))
        self.size = [1] * n
        self.components = n

    def find(self, x):
        # Path compression: make x point closer to the root.
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x

    def union(self, a, b):
        root_a, root_b = self.find(a), self.find(b)
        if root_a == root_b:
            return False

        # Attach the smaller component under the larger one.
        if self.size[root_a] < self.size[root_b]:
            root_a, root_b = root_b, root_a

        self.parent[root_b] = root_a
        self.size[root_a] += self.size[root_b]
        self.components -= 1
        return True


def component_count_dsu(n, edges):
    dsu = DSU(n)
    for a, b in edges:
        dsu.union(a, b)
    return dsu.components


assert component_count_dsu(5, [(0, 1), (1, 2), (3, 4)]) == 2
assert component_count_dsu(4, []) == 4

## Problem 15 — Minimum Spanning Tree with Kruskal (Medium/Hard)

Given a connected, undirected, weighted graph, return the total weight of its minimum spanning tree. Return `None` if the graph is disconnected.

In [ ]:
def kruskal_mst_weight(n, edges):
    dsu = DSU(n)
    total_weight = 0
    used_edges = 0

    for weight, u, v in sorted(edges):
        if dsu.union(u, v):
            total_weight += weight
            used_edges += 1
            if used_edges == n - 1:
                return total_weight

    return 0 if n == 1 else None


assert kruskal_mst_weight(4, [(1, 0, 1), (4, 0, 2), (2, 1, 2), (3, 1, 3), (5, 2, 3)]) == 6
assert kruskal_mst_weight(3, [(1, 0, 1)]) is None

# Module 6 — Shortest Paths with Constraints

## Problem 16 — Cheapest Flight with at Most K Stops (Medium)

Given directed flights `(from, to, price)`, find the cheapest cost from source to destination using at most `k` stops. Return `-1` if impossible.

The number of edges allowed is at most `k + 1`. A bounded Bellman-Ford approach uses a copy of the previous iteration so each round adds at most one edge.

In [ ]:
def cheapest_flight(n, flights, source, destination, k):
    infinity = float("inf")
    distance = [infinity] * n
    distance[source] = 0

    # Up to k stops means at most k+1 edges.
    for _ in range(k + 1):
        next_distance = distance.copy()
        for u, v, price in flights:
            if distance[u] != infinity:
                next_distance[v] = min(next_distance[v], distance[u] + price)
        distance = next_distance

    return -1 if distance[destination] == infinity else distance[destination]


assert cheapest_flight(4, [(0, 1, 100), (1, 2, 100), (2, 3, 100), (0, 3, 500)], 0, 3, 1) == 500
assert cheapest_flight(3, [(0, 1, 100), (1, 2, 100)], 0, 2, 1) == 200

# Module 7 — Tries and Bitmask Dynamic Programming

## Problem 17 — Trie Prefix Search (Medium)

A trie stores strings by shared prefixes. Implement insertion, exact search, and prefix search.

In [ ]:
class TrieNode:
    def __init__(self):
        self.children = {}
        self.is_word = False


class Trie:
    def __init__(self):
        self.root = TrieNode()

    def insert(self, word):
        node = self.root
        for char in word:
            if char not in node.children:
                node.children[char] = TrieNode()
            node = node.children[char]
        node.is_word = True

    def _walk(self, text):
        node = self.root
        for char in text:
            if char not in node.children:
                return None
            node = node.children[char]
        return node

    def search(self, word):
        node = self._walk(word)
        return node is not None and node.is_word

    def starts_with(self, prefix):
        return self._walk(prefix) is not None


trie = Trie()
trie.insert("apple")
assert trie.search("apple") is True
assert trie.search("app") is False
assert trie.starts_with("app") is True

## Problem 18 — Traveling Salesperson on a Small Graph (Hard)

For a small complete weighted graph, find the minimum cost route that starts at vertex 0, visits every vertex exactly once, and returns to vertex 0.

This is a bitmask-DP exercise. It is exponential, so it is intended for small `n`, not large real-world graphs.

In [ ]:
def tsp_bitmask(cost):
    n = len(cost)
    if n == 0:
        return 0
    if n == 1:
        return 0

    # dp[mask][last] = minimum cost to start at 0,
    # visit the vertices in mask, and finish at last.
    size = 1 << n
    inf = float("inf")
    dp = [[inf] * n for _ in range(size)]
    dp[1][0] = 0

    for mask in range(size):
        if not (mask & 1):
            continue
        for last in range(n):
            if dp[mask][last] == inf:
                continue
            for nxt in range(n):
                if mask & (1 << nxt):
                    continue
                new_mask = mask | (1 << nxt)
                dp[new_mask][nxt] = min(
                    dp[new_mask][nxt],
                    dp[mask][last] + cost[last][nxt]
                )

    full = size - 1
    return min(dp[full][last] + cost[last][0] for last in range(1, n))


matrix = [
    [0, 10, 15, 20],
    [10, 0, 35, 25],
    [15, 35, 0, 30],
    [20, 25, 30, 0]
]
assert tsp_bitmask(matrix) == 80

**Complexity:** O(n² · 2ⁿ) time and O(n · 2ⁿ) space.

# Module 8 — Interval DP

Interval DP solves a problem for every range `[left, right]`, often by trying a final split point or final operation.

## Problem 19 — Minimum Cost to Cut a Stick (Hard)

A stick has length `n`. You must make cuts at specified positions. The cost of each cut equals the length of the current stick segment being cut. Return the minimum total cost.

**Suggested LeetCode practice:** Minimum Cost to Cut a Stick.

In [ ]:
def min_cost_to_cut_stick(n, cuts):
    points = [0] + sorted(cuts) + [n]
    m = len(points)
    dp = [[0] * m for _ in range(m)]

    # gap is the distance between boundary indices.
    for gap in range(2, m):
        for left in range(m - gap):
            right = left + gap
            best = float("inf")

            for cut in range(left + 1, right):
                cost = points[right] - points[left] + dp[left][cut] + dp[cut][right]
                best = min(best, cost)

            dp[left][right] = 0 if best == float("inf") else best

    return dp[0][m - 1]


assert min_cost_to_cut_stick(7, [1, 3, 4, 5]) == 16
assert min_cost_to_cut_stick(9, []) == 0

# Module 9 — Hard Interview Pattern: Sweep Line and Events

Sweep-line algorithms sort events and process them in order. They are useful for overlapping intervals, meeting rooms, skyline-like problems, and resource allocation.

## Problem 20 — Minimum Meeting Rooms (Medium)

Given meeting intervals, return the minimum number of rooms required. Meetings that end at time `t` free a room for meetings starting at `t`.

In [ ]:
def min_meeting_rooms(intervals):
    if not intervals:
        return 0

    starts = sorted(start for start, end in intervals)
    ends = sorted(end for start, end in intervals)
    start_i = end_i = 0
    rooms = best = 0

    while start_i < len(starts):
        if starts[start_i] < ends[end_i]:
            rooms += 1
            best = max(best, rooms)
            start_i += 1
        else:
            rooms -= 1
            end_i += 1

    return best


assert min_meeting_rooms([[0, 30], [5, 10], [15, 20]]) == 2
assert min_meeting_rooms([[7, 10], [2, 4]]) == 1
assert min_meeting_rooms([]) == 0

# Module 10 — Hard Interview Pattern: Monotonic Queue

A monotonic deque keeps candidates in sorted order while the window moves. It can answer sliding-window maximum queries in O(n).

## Problem 21 — Sliding Window Maximum (Hard)

For every window of size `k`, return its maximum.

**Suggested LeetCode practice:** Sliding Window Maximum.

In [ ]:
from collections import deque


def sliding_window_maximum(nums, k):
    if not nums or not 1 <= k <= len(nums):
        raise ValueError("k must be between 1 and len(nums)")

    candidates = deque()  # Indices, values decreasing from front to back.
    answer = []

    for i, value in enumerate(nums):
        # Remove indices that have left the window.
        while candidates and candidates[0] <= i - k:
            candidates.popleft()

        # New value dominates smaller values behind it.
        while candidates and nums[candidates[-1]] <= value:
            candidates.pop()

        candidates.append(i)

        if i >= k - 1:
            answer.append(nums[candidates[0]])

    return answer


assert sliding_window_maximum([1, 3, -1, -3, 5, 3, 6, 7], 3) == [3, 3, 5, 5, 6, 7]
assert sliding_window_maximum([1], 1) == [1]

# Module 11 — Divide and Conquer

Divide and conquer splits a problem into smaller independent problems, solves them, and combines the results.

## Problem 22 — Count Inversions (Hard)

An inversion is a pair `(i, j)` where `i < j` but `nums[i] > nums[j]`. Count inversions efficiently using merge sort.

In [ ]:
def count_inversions(nums):
    values = list(nums)

    def sort_and_count(left, right):
        if right - left <= 1:
            return 0

        mid = (left + right) // 2
        count = sort_and_count(left, mid) + sort_and_count(mid, right)
        merged = []
        i, j = left, mid

        while i < mid and j < right:
            if values[i] <= values[j]:
                merged.append(values[i])
                i += 1
            else:
                merged.append(values[j])
                count += mid - i
                j += 1

        merged.extend(values[i:mid])
        merged.extend(values[j:right])
        values[left:right] = merged
        return count

    return sort_and_count(0, len(values))


assert count_inversions([2, 4, 1, 3, 5]) == 3
assert count_inversions([5, 4, 3, 2, 1]) == 10
assert count_inversions([]) == 0

**Complexity:** O(n log n) time and O(n) auxiliary space.

# Module 12 — Practice Set Without Immediate Solutions

Attempt these before looking up solutions. For each, write:
- a brute-force approach;
- the improved pattern;
- correctness reasoning;
- time and space complexity;
- at least three edge cases.

| # | Suggested problem | Pattern |
|---|---|---|
| 1 | Trapping Rain Water | Two pointers / monotonic stack |
| 2 | Maximal Rectangle | Histogram stack + row transformation |
| 3 | Median of Two Sorted Arrays | Binary search on partition |
| 4 | Find Minimum in Rotated Sorted Array | Modified binary search |
| 5 | Word Ladder | BFS / shortest path |
| 6 | Network Delay Time | Dijkstra |
| 7 | Critical Connections in a Network | DFS low-link values |
| 8 | Accounts Merge | Union-Find / graph |
| 9 | Burst Balloons | Interval DP |
| 10 | Regular Expression Matching | 2D dynamic programming |
| 11 | N-Queens | Backtracking + constraint sets |
| 12 | Word Search II | Trie + DFS backtracking |
| 13 | Largest Rectangle in Histogram | Monotonic stack |
| 14 | Sliding Window Maximum | Monotonic deque |
| 15 | Minimum Window Substring | Sliding window + counts |

# Complexity Quick Reference

| Pattern / algorithm | Typical time | Typical extra space |
|---|---:|---:|
| Monotonic stack | O(n) | O(n) |
| Binary search on answer | O(n log R) if feasibility is O(n) | O(1) beyond feasibility |
| 1D coin-change DP | O(amount × coin types) | O(amount) |
| LCS / edit distance | O(mn) | O(mn), or optimized for some versions |
| BFS / DFS | O(V + E) | O(V + E) graph storage |
| Dijkstra with heap | O((V + E) log V) | O(V + E) |
| Bellman-Ford | O(VE) | O(V) beyond edge storage |
| Kruskal MST | O(E log E) | O(V) DSU beyond sorting |
| Trie insert/search | O(length of word) | Depends on stored prefixes |
| Bitmask TSP DP | O(n² · 2ⁿ) | O(n · 2ⁿ) |
| Interval DP | Often O(n³) | O(n²) |
| Merge-sort inversion count | O(n log n) | O(n) |

# Final Readiness Checklist

Before calling yourself interview-ready at Medium/Hard level, aim to:

- [ ] Recognize a pattern from the constraints and wording.
- [ ] Explain a brute-force approach and its bottleneck.
- [ ] Derive the optimized algorithm instead of memorizing it.
- [ ] Explain correctness with an invariant or recurrence.
- [ ] State time and auxiliary-space complexity accurately.
- [ ] Handle duplicates, empty input, one-element input, disconnected graphs, and impossible cases.
- [ ] Know when an algorithm's assumptions fail (for example, Dijkstra with negative weights).
- [ ] Re-solve previously missed problems without notes.
- [ ] Practice explaining your solution aloud in 2–5 minutes.

**Recommended study rhythm:** 1–2 new problems per day, followed by review of older problems. For Hard problems, it is normal to need several sessions before the pattern becomes familiar.